In [2]:
# q3_gan_mnist.py

import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os

# Create output directory
os.makedirs('generated_images', exist_ok=True)

# Load MNIST dataset
(x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
x_train = (x_train - 127.5) / 127.5  # Normalize to [-1, 1]
x_train = x_train.reshape(-1, 28, 28, 1)

BUFFER_SIZE = 60000
BATCH_SIZE = 256

# Create dataset
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

# Generator model
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(7*7*256, use_bias=False, input_shape=(100,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((7, 7, 256)),
        layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')
    ])
    return model

# Discriminator model
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

# Loss and optimizers
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

generator = build_generator()
discriminator = build_discriminator()

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

# Training parameters
EPOCHS = 50
noise_dim = 100
num_examples_to_generate = 16

# Seed for evaluation
seed = tf.random.normal([num_examples_to_generate, noise_dim])

# Training step
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

# Save generated images
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.savefig(f'generated_images/image_at_epoch_{epoch:04d}.png')
    plt.close()

# Training loop
gen_losses = []
disc_losses = []

def train(dataset, epochs):
    for epoch in range(epochs):
        print(f'Starting epoch {epoch+1}/{epochs}...')

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)

        gen_losses.append(gen_loss)
        disc_losses.append(disc_loss)

        # Save sample images
        if epoch in [0, 49, 99]:
            generate_and_save_images(generator, epoch + 1, seed)

        print(f'Epoch {epoch+1}: Generator Loss: {gen_loss:.4f}, Discriminator Loss: {disc_loss:.4f}')

    # Plot loss curves
    plt.plot(gen_losses, label='Generator Loss')
    plt.plot(disc_losses, label='Discriminator Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('loss_plot.png')
    plt.close()

train(dataset, EPOCHS)


Starting epoch 1/50...
Epoch 1: Generator Loss: 0.8641, Discriminator Loss: 1.0351
Starting epoch 2/50...
Epoch 2: Generator Loss: 0.9932, Discriminator Loss: 1.0495
Starting epoch 3/50...
Epoch 3: Generator Loss: 1.0332, Discriminator Loss: 1.0977
Starting epoch 4/50...
Epoch 4: Generator Loss: 0.9605, Discriminator Loss: 1.2292
Starting epoch 5/50...
Epoch 5: Generator Loss: 0.8556, Discriminator Loss: 1.1914
Starting epoch 6/50...
Epoch 6: Generator Loss: 0.9079, Discriminator Loss: 1.1570
Starting epoch 7/50...
Epoch 7: Generator Loss: 1.1259, Discriminator Loss: 1.0120
Starting epoch 8/50...
Epoch 8: Generator Loss: 0.9366, Discriminator Loss: 1.1795
Starting epoch 9/50...
Epoch 9: Generator Loss: 0.8280, Discriminator Loss: 1.2742
Starting epoch 10/50...
Epoch 10: Generator Loss: 1.0296, Discriminator Loss: 1.0717
Starting epoch 11/50...
Epoch 11: Generator Loss: 0.8920, Discriminator Loss: 1.1529
Starting epoch 12/50...
Epoch 12: Generator Loss: 0.7744, Discriminator Loss: 1.380